# Compare the performances between base model and LoRA model
Qwen3-4B-Instruct with Flash-attention

## 1. Preparation

### 1.1 Import libraries

In [1]:
import os
import gc
from datetime import datetime
import json
import ast
import re
from tqdm import tqdm
import torch
from datasets import load_dataset, DatasetDict
from peft import PeftModel

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    set_seed
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

from transformers.utils.notebook import NotebookProgressCallback

In [2]:
import warnings
warnings.filterwarnings('ignore')

### 1.2 Set constants

In [3]:
MODEL_NAME = "../models/Qwen/Qwen3-4B-Instruct-2507"
DATA_PATH = "../data/final/LoRA_samples_sum_fixed.jsonl"

MAX_LENGTH = 2048
SEED = 42

set_seed(SEED)

### 1.3 Create dataset

Load and split the dataset with exactly the same approach as the training. This is to make sure we are using the same test set without data leakage.

In [4]:
ds_raw = load_dataset("json", data_files=DATA_PATH)["train"]
shuffled_ds = ds_raw.shuffle(seed=SEED)

In [5]:
train_temp = shuffled_ds.train_test_split(test_size=0.25, seed=SEED)

val_test = train_temp['test'].train_test_split(test_size=0.4, seed=42)

ds = DatasetDict({
    'train': train_temp['train'], # 75%
    'validation': val_test['train'], # 15%
    'test': val_test['test'] # 10%
})

Set up tokenizer and tokenized datasets with exactly the same approach as the training.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

In [7]:
tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

Set up the process function for correct tokenization for the data samples.

In [8]:
def process_func(example):

    messages = example["messages"]

    instruction_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    ) + tokenizer.eos_token

    tokenized_full = tokenizer(
        full_text,
        max_length=MAX_LENGTH,
        truncation=True,
        add_special_tokens=False
    )

    tokenized_instruction = tokenizer(
        instruction_text,
        max_length=MAX_LENGTH,
        truncation=True,
        add_special_tokens=False
    )

    input_ids = tokenized_full["input_ids"]
    attention_mask = tokenized_full["attention_mask"]

    instruction_len = len(tokenized_instruction["input_ids"])

    labels = [-100] * instruction_len + input_ids[instruction_len:]
    labels = labels[:len(input_ids)]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [9]:
tokenized_ds = ds.map(
    process_func,
    remove_columns=ds["train"].column_names,
    num_proc=os.cpu_count(),
    desc="Processing dataset"
)

In [10]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100
)

### 1.4 Helper functions

In [11]:
def extract_intention_id(response_text):
    """Extract intention_id with flexible JSON parsing"""
    response_text = response_text.strip()
    
    # Try 1: Standard JSON (double quotes)
    try:
        data = json.loads(response_text)
        if isinstance(data, dict):
            return data.get("intention_id", None)
    except:
        pass
    
    # Try 2: Python dict literal (handles single quotes)
    try:
        data = ast.literal_eval(response_text)
        if isinstance(data, dict):
            return data.get("intention_id", None)
    except:
        pass
    
    # Try 3: Regex fallback
    match = re.search(r'"intention_id"\s*:\s*"([^"]+)"', response_text)
    if not match:
        match = re.search(r"'intention_id'\s*:\s*'([^']+)'", response_text)
    if match:
        return match.group(1)
    
    return None


def parse_ground_truth(content):
    """Parse ground truth JSON with flexible quote handling"""
    content = content.strip()
    
    # Try 1: Standard JSON
    try:
        return json.loads(content)
    except:
        pass
    
    # Try 2: Python dict literal (single quotes)
    try:
        return ast.literal_eval(content)
    except:
        pass
    
    return None

In [12]:
def evaluate_intention_accuracy(model, tokenizer, test_data, name, batch_size=2):
    import json, ast, re, torch
    from tqdm import tqdm
    
    print(f"\nEvaluating {name} on intention_id...")
    
    # ✅ CRITICAL: Check test_data upfront
    print(f"  test_data type: {type(test_data)}")
    print(f"  test_data length: {len(test_data)}")
    if len(test_data) == 0:
        print("⚠️ ERROR: test_data is empty!")
        return {"accuracy": 0, "correct": 0, "total": 0, "skipped": 0}
    
    model.eval()
    
    # Save and set padding
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    correct = 0
    total = 0
    invalid_responses = 0
    samples_skipped = 0
    
    # Track skip reasons
    skip_reasons = {"no_messages": 0, "json_parse": 0, "no_id": 0, "prompt_gen": 0, "other": 0}
    
    for i in tqdm(range(0, len(test_data), batch_size), desc="Evaluating"):
        batch_start = i
        batch_end = min(i + batch_size, len(test_data))
        
        prompts = []
        ground_truth_ids = []
        
        for idx in range(batch_start, batch_end):
            try:
                example = test_data[idx]
                messages = example.get("messages", [])
                
                # Check 1: Messages structure
                if not messages or len(messages) < 2:
                    skip_reasons["no_messages"] += 1
                    samples_skipped += 1
                    continue
                
                # Check 2: Parse ground truth
                last_msg_content = messages[-1].get("content", "").strip()
                
                ground_truth_json = None
                try:
                    ground_truth_json = json.loads(last_msg_content)
                except:
                    try:
                        ground_truth_json = ast.literal_eval(last_msg_content)
                    except Exception as e:
                        skip_reasons["json_parse"] += 1
                        samples_skipped += 1
                        # Print first 3 JSON errors
                        if samples_skipped <= 3:
                            print(f"⚠️ JSON error (sample {idx}): {e}")
                        continue
                
                gt_id = ground_truth_json.get("intention_id") if ground_truth_json else None
                if gt_id is None:
                    skip_reasons["no_id"] += 1
                    samples_skipped += 1
                    continue
                
                # Check 3: Generate prompt
                try:
                    prompt = tokenizer.apply_chat_template(
                        messages[:-1],
                        tokenize=False,
                        add_generation_prompt=True
                    )
                except Exception as e:
                    skip_reasons["prompt_gen"] += 1
                    samples_skipped += 1
                    if samples_skipped <= 3:
                        print(f"⚠️ Prompt error (sample {idx}): {e}")
                    continue
                
                prompts.append(prompt)
                ground_truth_ids.append(str(gt_id))
                
            except Exception as e:
                skip_reasons["other"] += 1
                samples_skipped += 1
                # ✅ CRITICAL: Print first 5 unexpected errors
                if samples_skipped <= 5:
                    print(f"⚠️ Unexpected error (sample {idx}): {type(e).__name__}: {e}")
                continue
        
        # Skip batch if no valid prompts
        if len(prompts) == 0:
            continue
        
        # Tokenize
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # Evaluate predictions
        for j, output in enumerate(outputs):
            prediction_text = tokenizer.decode(
                output[inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            ).strip()
            
            predicted_id = extract_intention_id(prediction_text)
            
            if predicted_id is None:
                invalid_responses += 1
            elif str(predicted_id) == str(ground_truth_ids[j]):
                correct += 1
            total += 1
            
            # Print first 3 predictions for verification
            if total <= 100:
                print(f"\n--- Prediction {total} ---")
                print(f"Expected: {ground_truth_ids[j]}")
                print(f"Predicted: {predicted_id}")
                print(f"Raw: {prediction_text[:100]}...")
    
    # Restore padding
    tokenizer.padding_side = original_padding_side
    
    # Print skip reasons
    print(f"\n🔍 Skip reasons: {skip_reasons}")
    
    accuracy = (correct / total) * 100 if total > 0 else 0
    valid_rate = ((total - invalid_responses) / total) * 100 if total > 0 else 0
    
    print(f"\n{'='*60}")
    print(f"{name} Results:")
    print(f"{'='*60}")
    print(f"  Total processed: {total}")
    print(f"  Samples skipped: {samples_skipped}")
    print(f"  Correct: {correct}/{total} ({accuracy:.2f}%)")
    print(f"  Valid responses: {valid_rate:.2f}%")
    print(f"{'='*60}")
    
    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "invalid": invalid_responses,
        "valid_rate": valid_rate,
        "skipped": samples_skipped,
        "skip_reasons": skip_reasons
    }

## 2. Test the base model

### 2.1 Load base model and configure the test

In [13]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2"
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [14]:
base_model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [15]:
eval_args = TrainingArguments(
    output_dir="./temp_eval",
    per_device_eval_batch_size=1,  # ⬇️ CRITICAL: Was likely 4 or 8
    dataloader_num_workers=0,       # ✅ Windows-safe
    bf16=True,                      # ✅ Match training precision
    report_to="none",
)

base_trainer = Trainer(
    model=base_model,
    args=eval_args,
    eval_dataset=tokenized_ds["test"],
    data_collator=data_collator
)

base_trainer.remove_callback(NotebookProgressCallback)

### 2.2 Perform the test

In [16]:
# Evaluate loss
print("Computing evaluation loss...")
base_metrics = base_trainer.evaluate()
base_loss = base_metrics["eval_loss"]
base_perplexity = torch.exp(torch.tensor(base_loss)).item()
print(f"Base Model - Loss: {base_loss:.4f}, Perplexity: {base_perplexity:.2f}")

Computing evaluation loss...
Base Model - Loss: 1.8264, Perplexity: 6.21


In [17]:
print("\nComputing generation accuracy...")

# ✅ Ensure left-padding for generation
original_padding = tokenizer.padding_side
tokenizer.padding_side = "left"

base_results = evaluate_intention_accuracy(
    base_model,
    tokenizer,
    ds["test"],  # Raw dataset
    "Base Qwen3-4B",
    batch_size=2  # ⬇️ Small batch for generation too
)

# Restore padding
tokenizer.padding_side = original_padding


Computing generation accuracy...

Evaluating Base Qwen3-4B on intention_id...
  test_data type: <class 'datasets.arrow_dataset.Dataset'>
  test_data length: 1011


Evaluating:   0%|          | 1/506 [00:03<28:05,  3.34s/it]


--- Prediction 1 ---
Expected: 1a2b3c4d5e6f7g8h
Predicted: 1a2b3c4d5e6f7g8h
Raw: {"input_summary": "用户不在佛山", "intention_id": "1a2b3c4d5e6f7g8h"}...

--- Prediction 2 ---
Expected: n0o1p2q3r4s5t6u7
Predicted: k7l8m9n0o1p2q3r4
Raw: {"input_summary": "用户发送不当言论", "intention_id": "k7l8m9n0o1p2q3r4"}...


Evaluating:   0%|          | 2/506 [00:05<22:36,  2.69s/it]


--- Prediction 3 ---
Expected: 9z8y7x6w5v4u3t2s
Predicted: 9z8y7x6w5v4u3t2s
Raw: {"input_summary": "用户询问收费", "intention_id": "9z8y7x6w5v4u3t2s"}...

--- Prediction 4 ---
Expected: 1a2b3c4d5e6f7g8h
Predicted: 1a2b3c4d5e6f7g8h
Raw: {"input_summary": "用户询问带猫入场", "intention_id": "1a2b3c4d5e6f7g8h"}...


...


--- Prediction 97 ---
Expected: others
Predicted: others
Raw: {"input_summary": "用户质疑活动合理性", "intention_id": "others"}...

--- Prediction 98 ---
Expected: 0j1k2l3m4n5o6p7q
Predicted: 0j1k2l3m4n5o6p7q
Raw: {"input_summary": "用户询问纪念品", "intention_id": "0j1k2l3m4n5o6p7q"}...


Evaluating:  10%|▉         | 50/506 [01:47<15:44,  2.07s/it]


--- Prediction 99 ---
Expected: k707070707070707
Predicted: k101010101010101
Raw: {"input_summary": "用户询问位置", "intention_id": "k101010101010101"}...

--- Prediction 100 ---
Expected: d4e5f6g7h8i9j0k1
Predicted: d4e5f6g7h8i9j0k1
Raw: {"input_summary": "用户询问门票", "intention_id": "d4e5f6g7h8i9j0k1"}...


Evaluating: 100%|██████████| 506/506 [18:08<00:00,  2.15s/it]


🔍 Skip reasons: {'no_messages': 0, 'json_parse': 0, 'no_id': 0, 'prompt_gen': 0, 'other': 0}

Base Qwen3-4B Results:
  Total processed: 1011
  Samples skipped: 0
  Correct: 776/1011 (76.76%)
  Valid responses: 100.00%


### 2.3 Save the base model performance

In [18]:
performance_base =  {
    "loss": base_loss,
    "perplexity": base_perplexity,
    "accuracy": base_results.get("accuracy", 0),
    "correct": base_results.get("correct", 0),
    "total": base_results.get("total", 0),
    "valid_rate": base_results.get("valid_rate", 0)
}

with open("performance_base.json", "w", encoding="utf-8") as f:
    json.dump(performance_base, f, ensure_ascii=False, indent=2)

## 3. Test the LoRA model

### 3.1 Load the LoRA adapter and configure the test

Load the best performance adapter during training: lr1e-4_3epoch

In [19]:
adapter_dir = "./lora_adapter_lr1e-4_3epoch"

In [20]:
print("\nLoading LoRA model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="cuda:0",
    trust_remote_code=True,
    attn_implementation="flash_attention_2"
)

lora_model = PeftModel.from_pretrained(
    base_model,
    adapter_dir,
    # ✅ Load adapter to CPU first to avoid GPU OOM
    torch_device="cpu",
)

# Now move the merged model to GPU
lora_model = lora_model.to("cuda:0")
lora_model.eval()

print(f"✅ LoRA model loaded from: {adapter_dir}")


Loading LoRA model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

✅ LoRA model loaded from: ./lora_adapter


In [21]:
eval_args = TrainingArguments(
    output_dir="./temp_eval",
    per_device_eval_batch_size=1,  # ✅ Memory-safe
    dataloader_num_workers=0,
    bf16=True,
    report_to="none",
)

lora_trainer = Trainer(
    model=lora_model,
    args=eval_args,
    eval_dataset=tokenized_ds["test"],
    data_collator=data_collator
)

lora_trainer.remove_callback(NotebookProgressCallback)

### 3.2 Perform the test

In [22]:
print("\nComputing evaluation loss...")

lora_metrics = lora_trainer.evaluate()
lora_loss = lora_metrics["eval_loss"]
lora_perplexity = torch.exp(torch.tensor(lora_loss)).item()
print(f"LoRA Model - Loss: {lora_loss:.4f}, Perplexity: {lora_perplexity:.2f}")


Computing evaluation loss...
LoRA Model - Loss: 0.1198, Perplexity: 1.13


In [23]:
print("\nComputing generation accuracy...")

# ✅ Ensure left-padding for generation
original_padding = tokenizer.padding_side
tokenizer.padding_side = "left"

lora_results = evaluate_intention_accuracy(
    lora_model,
    tokenizer,
    ds["test"],  # Raw dataset with messages
    "LoRA Qwen3-4B",
    batch_size=2  # Memory-safe
)

# Restore padding
tokenizer.padding_side = original_padding


Computing generation accuracy...

Evaluating LoRA Qwen3-4B on intention_id...
  test_data type: <class 'datasets.arrow_dataset.Dataset'>
  test_data length: 1011


Evaluating:   0%|          | 1/506 [00:03<27:39,  3.29s/it]


--- Prediction 1 ---
Expected: 1a2b3c4d5e6f7g8h
Predicted: 1a2b3c4d5e6f7g8h
Raw: {"input_summary": "用户表示不在", "intention_id": "1a2b3c4d5e6f7g8h"}...

--- Prediction 2 ---
Expected: n0o1p2q3r4s5t6u7
Predicted: others
Raw: {"input_summary": "用户性骚扰客服", "intention_id": "others"}...


Evaluating:   0%|          | 2/506 [00:06<27:32,  3.28s/it]


--- Prediction 3 ---
Expected: 9z8y7x6w5v4u3t2s
Predicted: 9z8y7x6w5v4u3t2s
Raw: {"input_summary": "用户询问门票", "intention_id": "9z8y7x6w5v4u3t2s"}...

--- Prediction 4 ---
Expected: 1a2b3c4d5e6f7g8h
Predicted: others
Raw: {"input_summary": "询问能否带猫", "intention_id": "others"}...


...


--- Prediction 97 ---
Expected: others
Predicted: others
Raw: {"input_summary": "用户辱骂助手", "intention_id": "others"}...

--- Prediction 98 ---
Expected: 0j1k2l3m4n5o6p7q
Predicted: 0j1k2l3m4n5o6p7q
Raw: {"input_summary": "用户询问纪念品", "intention_id": "0j1k2l3m4n5o6p7q"}...


Evaluating:  10%|▉         | 50/506 [02:48<25:18,  3.33s/it]


--- Prediction 99 ---
Expected: k707070707070707
Predicted: k707070707070707
Raw: {"input_summary": "询问老场馆", "intention_id": "k707070707070707"}...

--- Prediction 100 ---
Expected: d4e5f6g7h8i9j0k1
Predicted: d4e5f6g7h8i9j0k1
Raw: {"input_summary": "用户询问门票", "intention_id": "d4e5f6g7h8i9j0k1"}...


Evaluating: 100%|██████████| 506/506 [26:58<00:00,  3.20s/it]


🔍 Skip reasons: {'no_messages': 0, 'json_parse': 0, 'no_id': 0, 'prompt_gen': 0, 'other': 0}

LoRA Qwen3-4B Results:
  Total processed: 1011
  Samples skipped: 0
  Correct: 941/1011 (93.08%)
  Valid responses: 100.00%


### 3.3 Save the LoRA model performance

In [24]:
performance_lora =  {
    "loss": lora_loss,
    "perplexity": lora_perplexity,
    "accuracy": lora_results.get("accuracy", 0),
    "correct": lora_results.get("correct", 0),
    "total": lora_results.get("total", 0),
    "valid_rate": lora_results.get("valid_rate", 0)
}

with open("performance_lora.json", "w", encoding="utf-8") as f:
    json.dump(performance_lora, f, ensure_ascii=False, indent=2)

## 4. Final comparison

In [ ]:
with open("performance_base.json", "r", encoding="utf-8") as f:
    performance_base = json.load(f)

In [25]:
performance_comparison = {
    "base_model": performance_base,
    "lora_model": performance_lora,
    "improvement": {
        "loss_reduction": (performance_base["loss"]-performance_lora["loss"])/performance_base["loss"],
        "perplexity_reduction": (performance_base["perplexity"]-performance_lora["perplexity"])/performance_base["perplexity"],
        "accuracy_gain": performance_lora['accuracy']-performance_base['accuracy'],
        "valid_rate_gain": performance_lora['valid_rate']-performance_base['valid_rate']
    }
}

In [26]:
with open("performance_comparison.json", "w", encoding='utf-8') as f:
    json.dump(performance_comparison, f, indent=2, ensure_ascii=False)

print(f"\n✅ Comparison results saved to: {"performance_comparison.json"}")


✅ Comparison results saved to: performance_comparison.json


In [27]:
# ----------------------------
# Compare Results
# ----------------------------
print("\n" + "=" * 60)
print("FINAL COMPARISON: Base vs LoRA")
print("=" * 60)
print(f"{'Metric':<25} {'Base Model':<15} {'LoRA Model':<15} {'Improvement':<15}")
print("-" * 60)
print(f"{'Eval Loss':<25} {performance_base["loss"]:.4f}{'':<10} {performance_lora["loss"]:.4f}{'':<10} {(performance_base["loss"]-performance_lora["loss"])/performance_base["loss"]*100:+.2f}%")
print(f"{'Perplexity':<25} {performance_base["perplexity"]:.2f}{'':<10} {performance_lora["perplexity"]:.2f}{'':<10} {(performance_base["perplexity"]-performance_lora["perplexity"])/performance_base["perplexity"]*100:+.2f}%")
print(f"{'Accuracy':<25} {performance_base["accuracy"]:.2f}%{'':<7} {performance_lora['accuracy']:.2f}%{'':<7} {performance_lora['accuracy']-performance_base['accuracy']:+.2f}%")
print(f"{'Valid Rate':<25} {performance_base["valid_rate"]:.2f}%{'':<7} {performance_lora['valid_rate']:.2f}%{'':<7} {performance_lora['valid_rate']-performance_base['valid_rate']:+.2f}%")
print("=" * 60)


FINAL COMPARISON: Base vs LoRA
Metric                    Base Model      LoRA Model      Improvement    
------------------------------------------------------------
Eval Loss                 1.8264           0.1198           +93.44%
Perplexity                6.21           1.13           +81.85%
Accuracy                  76.76%        93.08%        +16.32%
Valid Rate                100.00%        100.00%        +0.00%


**As show in the table, the LoRA model increased the accuracy from fairly risky 76.76% to the production grade of 93.08%.**  
This is a clear proof of the success of the fine-tuning to make the model more specialized for intention identification.